# Adversarial Attacks on Post-hoc Explainability Methods

We reimplement [Fooling LIME and SHAP: Adversarial Attacks on Post-hoc Explainability Methods](https://github.com/dylan-slack/Fooling-LIME-SHAP/tree/master).

One important consideration is going to be how we implement Local ALE for classification. We should think carefully about the specific model being learned.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def get_and_preprocess_compas_data(params):
	"""
    Handle processing of COMPAS according to: https://github.com/propublica/compas-analysis
	
	Parameters
	----------
	params : Params

	Returns
	----------
	Pandas data frame X of processed data, np.ndarray y, and list of column names
	"""
	PROTECTED_CLASS = params.protected_class
	POSITIVE_OUTCOME = params.positive_outcome
	NEGATIVE_OUTCOME = params.negative_outcome

	compas_df = pd.read_csv("data/compas.csv", index_col=0)

    # replicate data cleaning from original COMPAS analysis
	compas_df = compas_df.loc[(compas_df['days_b_screening_arrest'] <= 30) &
							  (compas_df['days_b_screening_arrest'] >= -30) &
							  (compas_df['is_recid'] != -1) &
							  (compas_df['c_charge_degree'] != "O") &
							  (compas_df['score_text'] != "NA")]
	compas_df['length_of_stay'] = (pd.to_datetime(compas_df['c_jail_out']) - pd.to_datetime(compas_df['c_jail_in'])).dt.days

    # reduce number of columns
	X = compas_df[['age', 'two_year_recid','c_charge_degree', 'race', 'sex', 'priors_count', 'length_of_stay']]

	# high scores (higher risk of recidivism) leads to negative outcomes
	y = np.array([NEGATIVE_OUTCOME if score == 'High' else POSITIVE_OUTCOME for score in compas_df['score_text']])
	sens = X.pop('race')

	# assign African-American as the protected class
	X = pd.get_dummies(X)
	sensitive_attr = np.array(pd.get_dummies(sens).pop('African-American'))
	X['race'] = sensitive_attr

	# make sure everything is lining up
	assert all((sens == 'African-American') == (X['race'] == PROTECTED_CLASS))
	cols = [col for col in X]
	
	return X, y, cols

## PCA Experiment

We show that the perturbations created by ALE are in some sense "milder" than the ones created by LIME and KernelSHAP.

In [ ]:
from adversarial_models import * 
from utils import *
from get_data import *

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd

import lime
import lime.lime_tabular
import shap

from copy import deepcopy


params = Params("model_configurations/experiment_params.json")
X, y, cols = get_and_preprocess_compas_data(params)
features = [c for c in X]

race_indc = features.index('race')

X = X.values
c_cols = [features.index('c_charge_degree_F'), features.index('c_charge_degree_M'), features.index('two_year_recid'), features.index('race'), features.index("sex_Male"), features.index("sex_Female")]

X = np.delete(X, c_cols, axis=1)

ss = StandardScaler().fit(X)
X = ss.transform(X)


r = []
for _ in range(1):
	p = np.random.normal(0,1,size=X.shape)
	
	# for row in p:
	# 	for c in c_cols:
	# 		row[c] = np.random.choice(X[:,c])

	X_p = X + p
	r.append(X_p)

r = np.vstack(r)
p = [1 for _ in range(len(r))]
iid = [0 for _ in range(len(X))]

all_x = np.vstack((r, X))
all_y = np.array(p + iid)

from matplotlib import pyplot as plt
from sklearn.decomposition import PCA

pca = PCA(n_components=2) 
results = pca.fit_transform(all_x)

print(len(X))

plt.scatter(results[:500,0], results[:500,1], alpha=.1)
plt.scatter(results[-500:,0], results[-500:,1], alpha=.1)
plt.show()




# COMPAS Experiment